# Activity 2: NYC Taxi ETL with pandas and SQLite

**Format:** Individual or pair practice  
**Estimated effort:** 50 to 65 minutes

This is an AI-Free Zone activity. Read only the stated public S3 object. Do not list the bucket, add AWS keys, or write anything to S3.

## Goal

Build a complete pandas ETL path: select source columns, read Parquet, inspect the result, validate the schema, derive fields, summarize the data, and publish two small summary tables to SQLite.

Write your solution in the TODO cells. After each major step, run the provided inspection cell and compare your result with the checkpoint output.


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

S3_URI = "s3://techcatalyst-de-2026/nyc-taxi/yellow-tripdata/yellow_tripdata_2026-01.parquet"

REQUIRED_COLUMNS = {
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID",
    "trip_distance", "fare_amount", "tip_amount", "payment_type",
}
CHARGE_COLUMNS = {
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "congestion_surcharge", "Airport_fee",
}

print("Source:", S3_URI)
print("Required columns:", len(REQUIRED_COLUMNS))
print("Charge columns:", len(CHARGE_COLUMNS))

PANDAS_STORAGE_OPTIONS = {"anon": True}

Source: s3://techcatalyst-de-2026/nyc-taxi/yellow-tripdata/yellow_tripdata_2026-01.parquet
Required columns: 8
Charge columns: 8


## Why `storage_options={"anon": True}` is required

The URI starts with `s3://`, so pandas passes the request to an S3 filesystem implementation. `storage_options` carries filesystem-specific settings to that implementation.

`anon=True` tells the S3 client to make an unsigned request. That is appropriate because this exact object permits public reads and does not require AWS credentials. Without anonymous mode, the client may search for credentials or sign the request, which can cause an access failure for this classroom source.

Anonymous access does not mean every S3 bucket is public. Keep the exact URI provided here.


## Part 1: Combine the two column contracts

Both `REQUIRED_COLUMNS` and `CHARGE_COLUMNS` are sets. Some names occur in both. Use a set union so each selected column appears once, then sort the result to give pandas a stable column order.

Create `read_columns` as a list with 14 unique names.

**Hint:** use either `REQUIRED_COLUMNS | CHARGE_COLUMNS` or `REQUIRED_COLUMNS.union(CHARGE_COLUMNS)`.


In [5]:
# TODO 1: union the two sets and sort the result.
read_columns = sorted(REQUIRED_COLUMNS.union(CHARGE_COLUMNS))

print("Columns to read:", len(read_columns))
read_columns


Columns to read: 14


['Airport_fee',
 'DOLocationID',
 'PULocationID',
 'congestion_surcharge',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'payment_type',
 'tip_amount',
 'tolls_amount',
 'tpep_dropoff_datetime',
 'tpep_pickup_datetime',
 'trip_distance']

### Expected checkpoint

`len(read_columns)` should be `14`. The ordered list begins with `Airport_fee`, `DOLocationID`, and `PULocationID`.


## Part 2: Extract and inspect

The read call is provided because the lesson is about the ETL workflow, not AWS authentication syntax.

- `columns=read_columns` reads only the 14 selected Parquet columns.
- `engine="pyarrow"` selects the Parquet engine explicitly.
- `storage_options=PANDAS_STORAGE_OPTIONS` sends anonymous-access settings to the S3 filesystem.

Run both inspection cells. `head()` answers, "What do sample rows look like?" `info()` answers, "How many rows and columns exist, what are their types, and roughly how much memory is used?"


In [10]:
pdf = pd.read_parquet(
    S3_URI,
    columns=read_columns,
    engine="pyarrow",
    storage_options=PANDAS_STORAGE_OPTIONS,
)
pdf.head()

,Airport_fee,DOLocationID,PULocationID,congestion_surcharge,extra,fare_amount,improvement_surcharge,mta_tax,payment_type,tip_amount,tolls_amount,tpep_dropoff_datetime,tpep_pickup_datetime,trip_distance
0,0.0,238,239,2.5,1.00,7.2,1.0,0.5,1,3.66,0.0,2026-01-01 00:59:37,2026-01-01 00:54:04,0.97
1,0.0,162,163,2.5,4.25,7.9,1.0,0.5,2,0.00,0.0,2026-01-01 00:39:47,2026-01-01 00:34:04,0.90
2,0.0,237,43,2.5,4.25,10.7,1.0,0.5,1,2.50,0.0,2026-01-01 01:05:59,2026-01-01 00:57:06,1.40
3,0.0,209,142,2.5,1.00,38.7,1.0,0.5,1,11.11,0.0,2026-01-01 00:58:10,2026-01-01 00:15:22,5.58
4,0.0,144,88,2.5,1.00,13.5,1.0,0.5,1,3.85,0.0,2026-01-01 00:40:43,2026-01-01 00:27:13,2.16


In [11]:
pdf.info()


<class 'pandas.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 14 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   Airport_fee            float64       
 1   DOLocationID           int32         
 2   PULocationID           int32         
 3   congestion_surcharge   float64       
 4   extra                  float64       
 5   fare_amount            float64       
 6   improvement_surcharge  float64       
 7   mta_tax                float64       
 8   payment_type           int64         
 9   tip_amount             float64       
 10  tolls_amount           float64       
 11  tpep_dropoff_datetime  datetime64[ns]
 12  tpep_pickup_datetime   datetime64[ns]
 13  trip_distance          float64       
dtypes: datetime64[ns](2), float64(9), int32(2), int64(1)
memory usage: 369.4 MB


### Expected extract output

- `pdf.head()` displays 5 rows and 14 columns. The first trip has pickup location `239`, dropoff location `238`, distance `0.97`, fare `7.20`, and tip `3.66`.
- `pdf.info()` reports `3,724,889` rows and 14 columns.
- The timestamp fields are `datetime64[ns]`. Location IDs are integers. Distance and charge fields are floating point.

If your row or column count differs, stop and verify the URI and `read_columns`.


## Part 3: Validate the schema contract

Create `missing_required` by subtracting the loaded column names from `REQUIRED_COLUMNS`. If the result is not empty, raise a `ValueError` that lists the missing names.

This check prevents the transformation from failing later with a less helpful missing-column error.


In [ ]:
# TODO 2: calculate missing_required and raise ValueError when needed.
# read_columns_set = set(read_columns)
missing_required = REQUIRED_COLUMNS - set(pdf.columns)

#missing_required = REQUIRED_COLUMNS.union(read_columns_set)

print("Missing required columns:", missing_required)


Missing required columns: set()


### Expected validation output

`Missing required columns: set()`


## Part 4: Transform

Create `pdf_clean` without removing rows. Add these four columns:

1. `trip_duration_minutes`: dropoff minus pickup, converted from seconds to fractional minutes.
2. `total_trip_charge`: the row-wise sum of every field in `CHARGE_COLUMNS`. Use a sorted list of the set when selecting pandas columns. Treat a missing charge component as zero only for this calculation.
3. `trip_date`: the date portion of `tpep_pickup_datetime`.
4. `is_valid_trip`: `True` when distance is at least 0 and duration is between 0 and 240 minutes, inclusive.

A trip lasting 5 minutes and 33 seconds is 5.55 minutes. Preserve the fraction so pandas and Polars use the same rule and produce matching averages.


In [22]:
total_trip_charge = pdf[list(CHARGE_COLUMNS)]
total_trip_charge

,tip_amount,improvement_surcharge,Airport_fee,mta_tax,fare_amount,extra,tolls_amount,congestion_surcharge
0,3.66,1.0,0.0,0.5,7.20,1.00,0.0,2.5
1,0.00,1.0,0.0,0.5,7.90,4.25,0.0,2.5
2,2.50,1.0,0.0,0.5,10.70,4.25,0.0,2.5
3,11.11,1.0,0.0,0.5,38.70,1.00,0.0,2.5
4,3.85,1.0,0.0,0.5,13.50,1.00,0.0,2.5
...,...,...,...,...,...,...,...,...
3724884,0.00,1.0,NaN,0.5,17.09,0.00,0.0,NaN
3724885,0.00,1.0,NaN,0.5,23.19,0.00,0.0,NaN
3724886,0.00,1.0,NaN,0.5,29.21,0.00,0.0,NaN
3724887,0.00,1.0,NaN,0.5,19.19,0.00,0.0,NaN


In [20]:
# TODO 3: create pdf_clean and all four derived columns.
# Useful methods: .assign(), .dt.total_seconds(), .fillna(0), .sum(axis=1),
# .dt.date, .ge(), and .between().

pdf["trip_duration_minutes"] = pdf["tpep_dropoff_datetime"] - (pdf["tpep_pickup_datetime"])

pdf["trip_duration_minutes"]


0         0 days 00:05:33
1         0 days 00:05:43
2         0 days 00:08:53
3         0 days 00:42:48
4         0 days 00:13:30
                ...      
3724884   0 days 00:13:16
3724885   0 days 00:00:14
3724886   0 days 00:15:47
3724887   0 days 00:09:39
3724888   0 days 00:14:33
Name: trip_duration_minutes, Length: 3724889, dtype: timedelta64[ns]

In [ ]:
display(pdf_clean.head())
pdf_clean.info()



### Expected transform output

- `pdf_clean.info()` reports `3,724,889` rows and 18 columns.
- The first five durations, rounded here, are `5.55`, `5.72`, `8.88`, `42.80`, and `13.50` minutes.
- The first five total charges are `15.86`, `16.15`, `21.45`, `54.81`, and `22.35`.
- The first five rows are valid trips.


## Part 5: Build two summaries

Create `pdf_daily_summary`, one row per `trip_date`, with:

- `trip_count`
- `average_distance`
- `average_duration_minutes`
- `total_trip_charge`
- `invalid_trip_count`

Sort by date. Then create `pdf_quality_summary`, a one-row DataFrame with `input_rows`, `output_rows`, `invalid_rows`, and `missing_fare_amount`.


In [ ]:
# TODO 4: create pdf_daily_summary and pdf_quality_summary.


In [ ]:
display(pdf_daily_summary.head())
pdf_daily_summary.info()
display(pdf_quality_summary)
pdf_quality_summary.info()


### Expected summary output

`pdf_daily_summary` has 33 rows and 6 columns.

| trip_date | trip_count | average_distance | average_duration_minutes | total_trip_charge | invalid_trip_count |
|---|---:|---:|---:|---:|---:|
| 2025-12-31 | 6 | 7.378333 | 17.258333 | 255.35 | 0 |
| 2026-01-01 | 114466 | 5.793117 | 15.268076 | 3426989.24 | 19 |
| 2026-01-02 | 100054 | 9.426764 | 16.619757 | 2856612.84 | 34 |
| 2026-01-03 | 108632 | 5.101376 | 15.844389 | 2992056.29 | 25 |
| 2026-01-04 | 93622 | 4.559145 | 15.630664 | 2709158.94 | 16 |

`pdf_quality_summary` should contain:

| input_rows | output_rows | invalid_rows | missing_fare_amount |
|---:|---:|---:|---:|
| 3724889 | 3724889 | 1544 | 0 |


## Part 6: Create the shared SQLite database

The code below creates an empty SQLite database in the same folder as this notebook. SQLite creates the file when `sqlite3.connect()` opens a path that does not exist.

Publish only the two small summaries. Writing all 3.7 million detail rows would make the refresher slower without adding value to the SQL handoff.


In [ ]:
DB_PATH = Path("nyc_taxi_refresher.db").resolve()

with sqlite3.connect(DB_PATH) as connection:
    connection.execute("SELECT 1")

print("SQLite database:", DB_PATH)

## Part 7: Publish the pandas tables

Use pandas `DataFrame.to_sql()` to write:

- `pdf_daily_summary` as `pandas_daily_summary`
- `pdf_quality_summary` as `pandas_quality_summary`

Open the connection with `sqlite3.connect(DB_PATH)`. Replace an existing table so the notebook can be rerun. Do not write the pandas index as a database column.


In [ ]:
# TODO 5: open a sqlite3 connection and write both pandas DataFrames.
# Hint: use to_sql(name=..., con=..., if_exists="replace", index=False).


In [ ]:
with sqlite3.connect(DB_PATH) as connection:
    sqlite_tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
        connection,
    )

sqlite_tables

### Expected SQLite checkpoint

After Activity 2, you should see:

- `pandas_daily_summary`
- `pandas_quality_summary`

Activity 3 adds two Polars tables to the same file.

## Verify in DBeaver

1. Copy the absolute database path printed above.
2. In DBeaver, choose **Database**, **New Database Connection**, then **SQLite**.
3. Paste the path into the SQLite database path field and finish the connection.
4. Expand the connection and refresh **Tables**.
5. Open both pandas tables. Confirm row counts of 33 and 1.


In [ ]:
assert len(pdf_clean) == len(pdf)
assert len(pdf_daily_summary) == 33
assert pdf_daily_summary["trip_count"].sum() == len(pdf_clean)
assert int(pdf_quality_summary.loc[0, "invalid_rows"]) == 1544
assert set(sqlite_tables["name"]) >= {"pandas_daily_summary", "pandas_quality_summary"}
print("pandas ETL and SQLite checks passed.")